**Intelligent Vision Systems · Lab 7 · Reading & Implementing Research Papers**
Companion to **Lecture 7 — Reading & Implementing Research Papers**
Dr. Muhammad Kazim · Department of Intelligent Systems, University of Lahore

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/09-paper-math-workbook.ipynb) &nbsp; [Lecture 7 slides](https://kazimbalti.github.io/intelligent-vision-systems/lectures/07_Paper_Implementation.html#/lab-7-and-resources) &nbsp; [All Lab 7 notebooks](https://kazimbalti.github.io/intelligent-vision-systems/labs.html#lab-7-papers) &nbsp; [Setup guide: Colab & Cursor](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/09_paper_implementation_setup.html)

# Lab 7I · Paper Math Workbook: Derive by Hand, Verify in Code

| ⏱ Time | 💻 Runs on | 🎯 You will build |
|---|---|---|
| the whole 3-hour session (run alongside the handbook) | Google Colab (CPU is enough, GPU is faster) or Cursor / VS Code on your laptop | the habit of every paper implementer: compute a small example with a pencil, then make the code reproduce it exactly |

**How to work.** Run the cells from top to bottom with **Shift + Enter**. Cells marked **🖐 Your turn** have a `# TODO`:
write a few lines, run the cell, then run the **✅ check** below it. A 💡 means "not yet": read the hint, fix, run again.
Stuck for more than 10 minutes? Open the **Hint** boxes, look at the matching lecture slide, or ask the AI assistant to
*explain* (not write) the step. **Use with the [Paper ↔ Code handbook](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/10_paper_code_side_by_side.html)**: each section here matches a Part of the handbook. Write your number in the `None`, run the cell, run the check.

**Rule of the workbook:** first the pencil, then the code. Every answer is a number (or a small list of numbers) you compute
by hand following the numbered derivation steps in the [handbook](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/10_paper_code_side_by_side.html). The `check(...)` cell then compares it with what
PyTorch computes. Two decimals are enough.

In [ ]:
# Setup: run this first (Colab already has PyTorch; on your laptop see requirements.txt).
# !pip install -q torch torchvision matplotlib scikit-learn opencv-python

import os, urllib.request
RAW = "https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/LAB/Lab_7"
for f in ["lab7_checks.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(f"{RAW}/{f}", f)
print("lab files ready")

In [ ]:
import math, time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import lab7_checks as C          # auto-checks + helpers for this lab

C.seed_everything(0)
device = C.get_device()
print("PyTorch", torch.__version__, "· device:", device)

In [ ]:
def check(name, yours, truth, tol=0.01):
    """Compare your hand-computed answer with the value the code computes."""
    if yours is None:
        print(f"⏳ {name}: write your answer first (replace None)."); return False
    y = torch.as_tensor(yours, dtype=torch.float64); t = torch.as_tensor(truth, dtype=torch.float64)
    if y.shape != t.shape:
        print(f"💡 {name}: your answer has shape {tuple(y.shape)}, the right one has {tuple(t.shape)}."); return False
    if torch.allclose(y, t, atol=tol, rtol=1e-3):
        print(f"✅ {name}: correct."); return True
    print(f"💡 {name}: not yet. Re-do the steps of the derivation on the handbook page, then try again."); return False
print("check() ready")

---
## 1 · Attention ([handbook Part 1](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/10_paper_code_side_by_side.html#attention))
The 3-word example: each row is one word.

### 📄 ↔ 💻 Paper ↔ Code: the equations you derive in this section

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, Eq. (1), §3.2.2, footnote 4</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_eq1.png" width="100%" alt="pc_t_eq1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_fn4.png" width="100%" alt="pc_t_fn4.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_mha.png" width="100%" alt="pc_t_mha.png"><br></td>
<td width="50%" valign="top">
<b>✏️ You derive by hand</b>
<ul>
<li><b>A1–A3</b>: Q Kᵀ, the softmax after ÷ √dₖ, and the output of Eq. (1) for 3 words</li>
<li><b>A4</b>: the same with the causal mask of §3.2.3</li>
<li><b>A5</b>: the standard deviation from footnote 4 (variance = dₖ)</li>
<li><b>A6</b>: parameters of MultiHead: 4 (d² + d)</li>
</ul>
<b>💻 Then the code checks you</b>: <code>check(name, yours, truth)</code> compares your number with PyTorch.
</td></tr>
</table>

In [ ]:
Q = torch.tensor([[1., 0], [0, 1], [1, 1]])
K = torch.tensor([[0., 1], [1, 0], [2, 0]])
V = torch.tensor([[10., 0], [0, 10], [5, 5]])

def attention(q, k, v, mask=None):
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))
    w = scores.softmax(-1)
    return w @ v, w

out, W = attention(Q, K, V)
print("we will check your hand results against these tensors (don't peek before computing!)")

**A1.** The raw scores $QK^\top$ for **word 1** (q₁ = (0, 1)), before scaling: a list of 3 numbers.

In [ ]:
A1 = None   # e.g. [?, ?, ?]

In [ ]:
check('A1 scores of word 1', A1, (Q @ K.T)[1])

**A2.** The softmax weights of **word 2** after scaling by √2 (3 numbers that sum to 1).

In [ ]:
A2 = None

In [ ]:
check('A2 weights of word 2', A2, W[2])

**A3.** The output of **word 1** (2 numbers).

In [ ]:
A3 = None

In [ ]:
check('A3 output of word 1', A3, out[1])

**A4.** With the **causal mask**, the weights of word 1 (3 numbers, the last one is 0).

In [ ]:
A4 = None

In [ ]:
_, Wc = attention(Q, K, V, torch.tril(torch.ones(3, 3, dtype=torch.bool)))
check('A4 causal weights of word 1', A4, Wc[1])

**A5.** The standard deviation of an **unscaled** dot product of two random vectors with d_k = 256 unit-variance entries (derivation C).

In [ ]:
A5 = None

In [ ]:
torch.manual_seed(0)
q, k = torch.randn(200000, 256), torch.randn(200000, 256)
measured = (q * k).sum(-1).std().item()
print(f'measured std = {measured:.2f}')
check('A5 std of q·k', A5, round(measured), tol=0.6)

**A6.** Parameters of one multi-head attention layer with d_model = 768 (derivation D).

In [ ]:
A6 = None

In [ ]:
check('A6 MHA parameters (d=768)', A6, sum(p.numel() for p in nn.MultiheadAttention(768, 12).parameters()), tol=0.5)

---
## 2 · Transformer blocks ([handbook Part 2](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/10_paper_code_side_by_side.html#transformer))

### 📄 ↔ 💻 Paper ↔ Code: the equations you derive in this section

<table>
<tr><th width="50%" align="left">📄 Paper · Vaswani et al. 2017, §3.1, §3.3, §3.5</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_pe.png" width="100%" alt="pc_t_pe.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_addnorm.png" width="100%" alt="pc_t_addnorm.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_t_ffn.png" width="100%" alt="pc_t_ffn.png"><br></td>
<td width="50%" valign="top">
<b>✏️ You derive by hand</b>
<ul>
<li><b>T1</b>: positional-encoding values from the sin / cos formulas of §3.5</li>
<li><b>T2</b>: LayerNorm of one token (mean and variance over its features)</li>
<li><b>T3–T4</b>: parameters of Eq. (2) and of a whole encoder layer</li>
</ul>
<b>💻 Then the code checks you</b> with the same <code>check(...)</code> cells.
</td></tr>
</table>

In [ ]:
def positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    pos = torch.arange(max_len).unsqueeze(1).float()
    div = 10000 ** (torch.arange(0, d_model, 2).float() / d_model)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos / div), torch.cos(pos / div)
    return pe
PE = positional_encoding(10, 4)

**T1.** PE for position 2 with d_model = 4 (4 numbers). Use sin 2 = 0.909, cos 2 = −0.416.

In [ ]:
T1 = None

In [ ]:
check('T1 PE(pos=2)', T1, PE[2])

**T2.** LayerNorm (γ = 1, β = 0) of the token x = (2, 4, 6, 8). Hint: μ = 5, variance = 5.

In [ ]:
T2 = None

In [ ]:
check('T2 LayerNorm(2,4,6,8)', T2, nn.LayerNorm(4, elementwise_affine=False)(torch.tensor([2., 4, 6, 8])))

**T3.** Parameters of the feed-forward block with d_model = 512, d_ff = 2048.

In [ ]:
T3 = None

In [ ]:
ffn = nn.Sequential(nn.Linear(512, 2048), nn.ReLU(), nn.Linear(2048, 512))
check('T3 FFN parameters', T3, sum(p.numel() for p in ffn.parameters()), tol=0.5)

**T4.** Parameters of one **encoder layer** of the base model (MHA + FFN + 2 LayerNorms, d = 512): derivation H, step 4.

In [ ]:
T4 = None

In [ ]:
layer = nn.TransformerEncoderLayer(512, 8, 2048)   # PyTorch's own encoder layer
check('T4 encoder layer parameters', T4, sum(p.numel() for p in layer.parameters()), tol=0.5)

---
## 3 · U-Net ([handbook Part 3](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/10_paper_code_side_by_side.html#unet))

### 📄 ↔ 💻 Paper ↔ Code: the text you turn into numbers in this section

<table>
<tr><th width="50%" align="left">📄 Paper · Ronneberger et al. 2015, §2, Figure 1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_u_arch.png" width="100%" alt="pc_u_arch.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_unet_fig1.png" width="100%" alt="p_unet_fig1.png"><br></td>
<td width="50%" valign="top">
<b>✏️ You derive by hand</b>
<ul>
<li><b>U1–U2</b>: output size out = n − 184 and the valid inputs n = 16k + 124</li>
<li><b>U3</b>: parameters of a convolution, k² C<sub>in</sub> C<sub>out</sub> + C<sub>out</sub></li>
<li><b>U4</b>: Dice and IoU from pixel counts</li>
</ul>
</td></tr>
</table>

In [ ]:
def unet2015_out(n):
    """Follow Figure 1 literally: 4 x (two valid convs, pool), bottom convs, 4 x (up-conv, two valid convs)."""
    s = n
    for _ in range(4):
        s -= 4
        assert s % 2 == 0, f"odd size {s} before a max-pool: invalid input"
        s //= 2
    s -= 4
    for _ in range(4):
        s = 2 * s - 4
    return s
print("unet2015_out(572) =", unet2015_out(572))

**U1.** Output size of the 2015 U-Net for a **540 × 540** input (derivation I: out = n − 184).

In [ ]:
U1 = None

In [ ]:
check('U1 output for 540', U1, unet2015_out(540), tol=0.5)

**U2.** The **smallest valid input larger than 600** (n = 16k + 124).

In [ ]:
U2 = None

In [ ]:
def ok(n):
    try:
        unet2015_out(n); return True
    except AssertionError:
        return False
check('U2 smallest valid input > 600', U2, min(n for n in range(601, 700) if ok(n)), tol=0.5)

**U3.** Parameters of the **first DoubleConv** (1 → 64 → 64, unpadded 3 × 3, with biases).

In [ ]:
U3 = None

In [ ]:
dc = nn.Sequential(nn.Conv2d(1, 64, 3), nn.ReLU(), nn.Conv2d(64, 64, 3), nn.ReLU())
check('U3 first DoubleConv parameters', U3, sum(p.numel() for p in dc.parameters()), tol=0.5)

**U4.** Dice and IoU (2 numbers) for: 50 predicted pixels, 50 true pixels, 40 overlapping.

In [ ]:
U4 = None

In [ ]:
pred = torch.zeros(100, dtype=torch.bool); true = torch.zeros(100, dtype=torch.bool)
pred[:50] = True; true[10:60] = True                       # overlap = 40 pixels
I, U = (pred & true).sum().item(), (pred | true).sum().item()
check('U4 Dice, IoU', U4, [2 * I / (pred.sum() + true.sum()).item(), I / U])

---
## 4 · Vision Transformer ([handbook Part 4](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/10_paper_code_side_by_side.html#vit))

### 📄 ↔ 💻 Paper ↔ Code: the equations you derive in this section

<table>
<tr><th width="50%" align="left">📄 Paper · Dosovitskiy et al. 2021, §3.1, Eq. (1)–(4), Table 1</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_eq.png" width="100%" alt="pc_v_eq.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_v_table1.png" width="100%" alt="pc_v_table1.png"><br></td>
<td width="50%" valign="top">
<b>✏️ You derive by hand</b>
<ul>
<li><b>V1</b>: N = HW / P² and the number of tokens (+ 1 for the [class] token)</li>
<li><b>V2</b>: the parameters of E, (P² C) · D + D</li>
<li><b>V3–V4</b>: parameter counts that reproduce Table 1 ("86M")</li>
</ul>
</td></tr>
</table>

**V1.** Number of tokens (patches + [class]) for ViT-B/16 on a **384 × 384** image.

In [ ]:
V1 = None

In [ ]:
conv = nn.Conv2d(3, 8, 16, stride=16)
check('V1 tokens at 384', V1, conv(torch.zeros(1, 3, 384, 384)).flatten(2).shape[-1] + 1, tol=0.5)

**V2.** Parameters of the ViT-B/16 **patch embedding** (P = 16, C = 3, D = 768).

In [ ]:
V2 = None

In [ ]:
check('V2 patch embedding parameters', V2, sum(p.numel() for p in nn.Conv2d(3, 768, 16, stride=16).parameters()), tol=0.5)

**V3.** Parameters of **one ViT-B encoder block** (2 LayerNorms + MSA + MLP 768 → 3072 → 768): derivation K, step 3.

In [ ]:
V3 = None

In [ ]:
import torchvision
vit = torchvision.models.vit_b_16()
check('V3 one ViT-B block', V3, sum(p.numel() for p in vit.encoder.layers[0].parameters()), tol=0.5)

**V4.** The whole of ViT-B/16 (derivation K): total parameters.

In [ ]:
V4 = None

In [ ]:
check('V4 ViT-B/16 total', V4, sum(p.numel() for p in vit.parameters()), tol=0.5)

---
## 5 · ResNet ([handbook Part 5](https://kazimbalti.github.io/intelligent-vision-systems/tutorials/10_paper_code_side_by_side.html#resnet))

### 📄 ↔ 💻 Paper ↔ Code: the equation you reason about in this section

<table>
<tr><th width="50%" align="left">📄 Paper · He et al. 2016, §3.2, Eq. (1)–(2), Figure 2</th><th width="50%" align="left">💻 Code</th></tr>
<tr><td width="50%" valign="top"><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/pc_r_eq1.png" width="100%" alt="pc_r_eq1.png"><br><img src="https://raw.githubusercontent.com/Kazimbalti/intelligent-vision-systems/main/lectures/assets/07_paper_implementation/p_resnet_fig2.png" width="100%" alt="p_resnet_fig2.png"><br></td>
<td width="50%" valign="top">
<b>✏️ You derive by hand</b>
<ul>
<li><b>R1</b>: the gradient factor after 20 plain layers with derivative 0.5 each</li>
<li><b>R2</b>: the same with shortcuts, y = F(x) + x, so ∂y/∂x = F′(x) + 1</li>
</ul>
</td></tr>
</table>

**R1.** Gradient factor after 20 plain layers whose derivative is 0.5 each (one number, use scientific notation like `9.5e-7`).

In [ ]:
R1 = None

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x
for _ in range(20):
    y = 0.5 * y                       # a 'layer' with derivative 0.5
y.backward()
check('R1 plain gradient', R1, x.grad, tol=1e-8)

**R2.** The same 20 layers with a shortcut, y = F(x) + x, where F has derivative 0 (an 'unused' layer).

In [ ]:
R2 = None

In [ ]:
x = torch.tensor(1.0, requires_grad=True)
y = x
for _ in range(20):
    y = 0.0 * y + y                   # F(x) = 0·x, plus the identity
y.backward()
check('R2 residual gradient', R2, x.grad)

---
## 🏁 Done?
Count your ✅. Then pick the derivation you found hardest and **explain it to your neighbour** using the handbook page,
without notes. Next: build the full models in [Lab 7C](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/03-transformer-reverse-sequence.ipynb),
[7D](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/04-unet-from-paper.ipynb) and
[7F](https://colab.research.google.com/github/Kazimbalti/intelligent-vision-systems/blob/main/LAB/Lab_7/06-vit-from-paper.ipynb).